# Adversarial Machine Learning attack against a DL-based NIDS
In this experiment, our primary focus is to assess the robustness of DL-based NIDSs against evasion attacks using AML. Specifically, we evaluate an MLP-based NIDS under two conditions: (1) unperturbed attack and benign network traffic, and (2) manipulated network traffic.

The objective of this study is to analyze how network traffic manipulation impacts the accuracy of the NIDS and to explore how attackers can exploit weaknesses in NIDSs to execute successful attacks.

The model is designed for binary classification of the network traffic as benign or malicious. Therefore, it returns a value between 0 and 1, which is the probability of the input flow of being malicious. 


| <img src="./mlp.jpg" width="100%">  |
|--|
| Sample MLP architecture|


The laboratory is divided into two phases:
-  **Phase 1**: MLP-based NIDS
    - train a pre-defined MLP model and test it using the test set and an unperturbed traffic trace [ddos-chunk-short.pcap](./DOS2019_Binary_5_Attacks_PCAPs/ddos-chunk-short.pcap)
    - ensure that the model achieves a high accuracy on the test set and that it correctly classifies the unperturbed attack traffic trace as malicious
    - if not, you can modify the model architecture and/or hyperparameters to improve the accuracy of the model on the test set and the unperturbed attack traffic trace
    - change the ```train``` and ```predict``` functions in the notebook to make them accept the dataset path and model path as arguments
    - convert the notebook into a Python script using the command ```jupyter nbconvert --to script NIDS_AML.ipynb```
-  **Phase 2**: Live AML attack  
    - run the NIDS in a separate terminal using the command ```python NIDS_AML.py --predict_live ./DOS2019_Binary_5_Attacks_PCAPs/ddos-chunk-short.pcap --model ./MLP_model.keras```. This verifies that the model correctly classifies the unperturbed attack traffic trace as malicious.
    - perturb the original traffic using the [aml-attack.py](./aml-attack.py) script, which adds one or more perturabations to the packets among the following:
        - Random TCP Window size (--min-tcp-window --max-tcp-window) using values between 0 and 65535
        - Random IAT perturbation (--min-delay-ms --max-delay-ms) using values between 1 and 1000 milliseconds
        - Random packet length change (--min-length-change --max-length-change) using values between 0 and 1500 bytes
        - Random TTL change (--min-ttl --max-ttl) using values between 0 and 255
        - IP fragmentation flags (--perturb-ip-flags)
    - Example command to perturb the IP flags::
    ```python aml-attack.py DOS2019_Binary_5_Attacks_PCAPs/ddos-chunk-short.pcap -o  DOS2019_Binary_5_Attacks_PCAPs/ddos-chunk-short-aml.pcap --perturb-ip-flags```
    - Finally run the NIDS in a separate terminal using the command ```python NIDS_AML.py --predict_live ./DOS2019_Binary_5_Attacks_PCAPs/ddos-chunk-short-aml.pcap --model ./MLP_model.keras```. This verifies that the model misclassifies the perturbed attack traffic trace as benign.

## Dataset
You will use a dataset of benign and various DDoS attacks from the CIC-DDoS2019 dataset (https://www.unb.ca/cic/datasets/ddos-2019.html).
The network traffic has been previously pre-processed in a way that packets are grouped in bi-directional traffic flows using the 5-tuple (source IP, destination IP, source Port, destination Port, protocol). Each flow is represented with 21 packet-header features computed from max 1000 packets:

| Feature nr.         | Feature Name |
|---------------------|---------------------|
| 00 | timestamp (mean IAT) | 
| 01 | packet_length (mean)| 
| 02 | IP_ttl (mean) |
| 03 | IP_flags_df (sum) |
| 04 | IP_flags_mf (sum) |
| 05 | IP_flags_rb (sum) | 
| 06 | IP_frag_off (sum) |
| 07 | TCP_length (mean) |
| 08 | TCP_flags_ack (sum) |
| 09 | TCP_flags_cwr (sum) |
| 10 | TCP_flags_ece (sum) |
| 11 | TCP_flags_fin (sum) |
| 12 | TCP_flags_push (sum) |
| 13 | TCP_flags_reset (sum) |
| 14 | TCP_flags_syn (sum) |
| 15 | TCP_flags_urg (sum) |
| 16 | TCP_window_size (mean) |
| 17 | UDP_length (mean) |
| 18 | ICMP_type (mean) |
| 19 | ICMP_code (mean) |
| 20 | Packets (counter)|

In [ ]:
# Author: Roberto Doriguzzi-Corin
# Project: Course on Network Intrusion and Anomaly Detection with Machine Learning
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import time
import argparse
import pyshark
import numpy as np
import pprint
from scipy.stats import uniform, randint
import tensorflow as tf
import matplotlib.pyplot as plt
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import RandomizedSearchCV
from tensorflow.keras.models import Sequential,load_model, save_model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Dense, Conv2D, GlobalMaxPooling2D, Flatten, Dropout
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.utils import set_random_seed
from lucid_dataset_parser import *
from util_functions import *

# We need the following to get around “RuntimeError: This event loop is already running” when using Pyshark within Jupyter notebooks.
# Not needed in stand-alone Python projects
import nest_asyncio
nest_asyncio.apply()  

EPOCHS = 1000
TEST_ITERATIONS=10

### SELECT THE MODEL_TYPE HERE ('MLP' or 'CNN') ###
MODEL_TYPE = 'MLP' 
###################################################

DATASET_PATH = './DOS2019_Binary_5_Attacks_' + MODEL_TYPE

# disable GPUs for test reproducibility
tf.config.set_visible_devices([], 'GPU')

SEED = 0
np.random.seed(SEED)
set_random_seed(SEED)

# Argument parser
The following cell defines the arguments that are accepted by the NIDS. The arguments ```--train``` and ```--predict``` can be used to indicate the folder with the dataset. The script will loaf the ```hdf5``` files for training and testing respectively. The argument ```--predict_live``` can be used to perform predictions on live traffic captured from a network interface (e.g., ```eth0``` or ```lo```) or to make prediction using a pre-recorded traffic trace (e.g, ```ddos-chunk.pcap```). In both cases the argument is a string. In the first case, it is the name of the interface, in the second case, the path to the ```pcap``` file. The argument ```--model``` indicates the path to a trained model that will be used to make predictions (```--predict``` or ```predict_live```). 

In [ ]:
parser = argparse.ArgumentParser(description="A DL-based NIDS for DDoS attack detection")
args = parser.parse_args(args=[])
parser.add_argument('-t', '--train', nargs='?', type=str,  default=None, help="Start the training process")
parser.add_argument('-p', '--predict', nargs='?', type=str,  default=None, help="Perform a prediction on pre-preprocessed data")
parser.add_argument('-pl', '--predict_live', nargs='?', type=str, default=None, help='Perform a prediction on live traffic or on a pre-recorded traffic trace in pcap format')
parser.add_argument('-m', '--model', type=str, default = None, help='File containing the model in keras format')

args, unknown = parser.parse_known_args()
print("see all args:", args)

In [ ]:
def report_results(Y_true, Y_pred, model_name, data_source, prediction_time):
    ddos_rate = '{:04.3f}'.format(sum(Y_pred) / Y_pred.shape[0])

    if Y_true is not None and len(Y_true.shape) > 0:  # if we have the labels, we can compute the classification accuracy
        Y_true = Y_true.reshape((Y_true.shape[0], 1))
        accuracy = accuracy_score(Y_true, Y_pred)

        f1 = f1_score(Y_true, Y_pred)
        tn, fp, fn, tp = confusion_matrix(Y_true, Y_pred, labels=[0, 1]).ravel()
        tnr = tn / (tn + fp)
        fpr = fp / (fp + tn)
        fnr = fn / (fn + tp)
        tpr = tp / (tp + fn)

        row = {'Model': model_name, 'Time': '{:04.3f}'.format(prediction_time),
               'Samples': Y_pred.shape[0], 'DDOS%': ddos_rate, 'Accuracy': '{:05.4f}'.format(accuracy), 'F1Score': '{:05.4f}'.format(f1),
               'TPR': '{:05.4f}'.format(tpr), 'FPR': '{:05.4f}'.format(fpr), 'TNR': '{:05.4f}'.format(tnr), 'FNR': '{:05.4f}'.format(fnr), 'Source': data_source}

        pprint.pprint(row, sort_dicts=False)

# Implement a Neural Network model
Finalise the CNN model by using all the four arguments of ```create_CNN_model```. You will use ALL these arguments to tune the model afterwards.

In [ ]:
def create_MLP_model(optimizer=Adam, dense_layers=4, hidden_units=2, learning_rate = 0.001):
    model = Sequential(name  = "mlp")
    model.add(Input(shape=(21,)))
    model.add(Dense(hidden_units, activation='relu'))
    for layer in range(dense_layers):
        model.add(Dense(hidden_units, activation='relu', name='hidden-fc' + str(layer)))
    model.add(Dense(1, activation='sigmoid', name='output'))
    model.compile(optimizer=optimizer(learning_rate=learning_rate), loss='binary_crossentropy', metrics=['accuracy'])
    model.summary()
    return model

# Function to create the CNN model
def create_CNN_model(optimizer=Adam, filters = 100, kernel_size=(3,3), strides=(1,1), padding='same',learning_rate = 0.001,dropout_rate=0.1):
    model = Sequential(name  = "cnn")

    model.add(Input(shape=(10,20,1)))
    model.add(Conv2D(filters=filters, kernel_size=kernel_size, data_format='channels_last', activation='relu', padding=padding, strides=strides))
    model.add(Dropout(dropout_rate))
    model.add(GlobalMaxPooling2D())
    model.add(Flatten())
    model.add(Dense(1, activation='sigmoid',name='output'))
    model.compile(optimizer=optimizer(learning_rate=learning_rate), loss='binary_crossentropy', metrics=['accuracy'])
    
    model.summary()
    return model

# Prediction on static test set

In [ ]:
def predict(dataset_path, model_path):
    if dataset_path is not None:
        X_test, y_test = load_dataset(dataset_path + "/*" + '-test.hdf5')

        if model_path == None or model_path.endswith('.keras') == False:
                print ("No valid model specified!")
                return

        if model_path is not None:
            model = load_model(model_path)
        else:
            print ("Invalid model path: ", model_path) 
            return

        pt0 = time.time()
        for i in range(TEST_ITERATIONS):
            Y_pred = np.squeeze(model.predict(X_test, batch_size=16) > 0.5,axis=1)
        pt1 = time.time()
        prediction_time = pt1 - pt0

        report_results(np.squeeze(y_test), Y_pred,  model.name, '', prediction_time/TEST_ITERATIONS)

# Prediction on live traffic

In [ ]:
def predict_live(source,model_path):
    if source is not None:
        if source.endswith('.pcap'):
            pcap_file = source
            cap = pyshark.FileCapture(pcap_file)
            data_source = pcap_file.split('/')[-1].strip()
        else:
            cap =  pyshark.LiveCapture(interface=source)
            data_source = args.predict_live

        print ("Prediction on network traffic from: ", source)

        if model_path is not None:
            model = load_model(model_path)
        else:
            print ("Invalid model path: ", model_path) 
            return

        # load the labels, if available
        labels = parse_labels('DOS2019')

        if MODEL_TYPE == 'MLP':
            MAX_FLOW_LEN=1000
            mins, maxs = static_min_max(flatten=True,time_window=10,max_flow_len=MAX_FLOW_LEN)
        else:
            MAX_FLOW_LEN=10
            mins, maxs = static_min_max(flatten=False,time_window=10,max_flow_len=MAX_FLOW_LEN)

        while (True):
            samples = process_live_traffic(cap, 'DOS2019', labels, max_flow_len=MAX_FLOW_LEN, traffic_type="all",time_window=10)
            if len(samples) > 0:
                X,Y_true,flow_ids = dataset_to_list_of_fragments(samples)
                if MODEL_TYPE == 'MLP':
                    X = flatten_samples(X)
                    X = np.array(normalize(X, mins, maxs))
                else:
                    X = np.array(normalize_and_padding(X, mins, maxs, MAX_FLOW_LEN))
                if labels is not None:
                    Y_true = np.array(Y_true)
                else:
                    Y_true = None
                
                pt0 = time.time()
                Y_pred = np.squeeze(model.predict(X, batch_size=2048) > 0.5,axis=1)
                pt1 = time.time()
                prediction_time = pt1 - pt0

                report_results(np.squeeze(Y_true), Y_pred,  MODEL_TYPE, '', prediction_time)

# Hyper-parameter tuning
Define the list of hyperparameters, along with their search ranges and distributions, for the CNN model.

In [ ]:
def train(dataset_path, model_path):

    if dataset_path is not None:
        X_train, y_train = load_dataset(dataset_path + "/*" + '-train.hdf5')
        X_val, y_val = load_dataset(dataset_path + "/*" + '-val.hdf5')

        param_mlp_dist = {
            'model__learning_rate' : uniform(0.001, 0.01),
            'optimizer' : [SGD,Adam],
            'model__dense_layers' : randint(1,8),
            'model__hidden_units': randint(2,32)
        }

        param_cnn_dist = {
            'model__learning_rate' : uniform(0.001, 0.01),
            'model__filters' : randint(4,16),
            'optimizer' : [SGD,Adam],
            'model__kernel_size': [(3,3),(2,3)],
            'model__strides': [(1,1),(2,2)],
            'model__padding' : ['same', 'valid']
        }

        if MODEL_TYPE == 'MLP':
            param_dist = param_mlp_dist
            create_model = create_MLP_model
        else:
            param_dist = param_cnn_dist
            create_model = create_CNN_model

        early_stopping = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10, restore_best_weights=True)
        model = KerasClassifier(model=create_model, batch_size=100, verbose=1,callbacks=[early_stopping])
        random_search = RandomizedSearchCV(estimator=model, param_distributions=param_dist, n_iter=3, cv=2, random_state=SEED)
        random_search_result = random_search.fit(X_train, y_train,epochs=EPOCHS, validation_data=(X_val, y_val), callbacks=[early_stopping])

        # Print the best parameters and corresponding accuracy
        print("Best parameters found: ", random_search_result.best_params_)
        print("Best cross-validated accuracy: {:.2f}".format(random_search_result.best_score_))

        # Save the best model
        best_model = random_search.best_estimator_.model_
        if model_path is not None:
            save_model(best_model,model_path)
        else:
            print ("Invalid model path: ", model_path)
            print ("Model saved as: ./" + MODEL_TYPE + '_model.keras')
            save_model(best_model, './'+ MODEL_TYPE + '_model.keras')

# Train your model
Train the model by calling the ```train``` method with static dataset path, which depends on the ```MODEL_TYPE``` variable, as the shape of the input layer of the two models is different (vector for the MLP, array for the CNN).

If you wish to export the notebook as a stand-alone Python script, replace the two arguments with ```args.train``` and ```args.model``` (see the lab on NIDS deployment for more info).

In [ ]:
# Train the model
train(DATASET_PATH, './' + MODEL_TYPE + '_model.keras')
#train(args.train,args.model)

# Make predictions on the test set
In the following cell, you can make prediction with test traffic samples obtained from unperturbed traffic traces. This test will give you a performance baseline for your model.

Also in this case, the dataset path depends on the ```MODEL_TYPE``` variable, as the shape of the input layer of the two models is different (vector for the MLP, array for the CNN).

If you wish to export the notebook as a stand-alone Python script, replace the two arguments with ```args.predict``` and ```args.model``` (see the lab on NIDS deployment for more info).

In [ ]:
# Predictions on the test set
predict(DATASET_PATH, './' + MODEL_TYPE + '_model.keras')
#predict(args.predict,args.model)

# Make predictions using a pcap file
In the following cell, you can test the NIDS using traces of network traffic. 
Start with unperturbed traffic (i.e., running the code below), then use perturbed traffic traces to evaluate the robustness of the NN model to evasion AML attacks.

Available traffic traces are:
- Clean traffic trace [ddos-chunk-short.pcap](./DOS2019_Binary_5_Attacks_PCAPs/ddos-chunk-short.pcap)
- Perturbed traffic trace [ddos-chunk-short-aml.pcap](./DOS2019_Binary_5_Attacks_PCAPs/ddos-chunk-short-aml.pcap)

**NOTE**: the important metric here is the FNR, which is the percentage of malicious samples that evades the NIDS.

If you wish to export the notebook as a stand-alone Python script, replace the two arguments with ```args.predict_live``` and ```args.model``` (see the lab on NIDS deployment for more info).


In [ ]:
# Predictions on a pcap file

### CHANGE THE PCAP PATH HERE ###
#predict_live('./DOS2019_Binary_5_Attacks_PCAPs/ddos-chunk-short.pcap','./' + MODEL_TYPE + '_model.keras')
#predict_live('./DOS2019_Binary_5_Attacks_PCAPs/ddos-chunk-short-aml.pcap','./' + MODEL_TYPE + '_model.keras')
predict_live(args.predict_live,args.model)
##########################